# T25b — LoRA adapter on bge-small-en-v1.5 for CallVerse Intake (Colab, GPU)

Written by the Brain, 2026-10-31, approved by Yassine (all ⭐). **Colab: Runtime → Change runtime type → T4 GPU.** Then *Run all*. Download `bge-small-callverse-lora.zip` at the end (a few MB) and put it in `models/` on the PC.

What it does: freezes `BAAI/bge-small-en-v1.5` (pinned revision used by the system) and trains only a **LoRA adapter** (r 16, alpha 32, dropout 0.1, on query / key / value) with the same label-based triplet loss as T23b. On the PC the adapter is **on for Intake and off for rule search**, so rule search stays exactly as today.

Training data: **ABCD `train`** ('other' capped at 500) + **Bitext Retail** (ShopNova) + **Bitext Travel** (Atlas), mapped to our reasons, 300 per reason max + **120 Brain-written Atlas messages** for reasons Bitext lacks. **Never used:** our 140 Intake index examples, ABCD `dev` / `test`, Atlas val / test, rule-search sets. Scoring happens on the PC in T25b with the T25 rule.

In [ ]:
!pip -q install "sentence-transformers>=3.3,<7" peft datasets
!pip -q uninstall -y torchao   # Colab ships an old torchao that makes peft fail; LoRA does not need it
import json, gzip, random, re, urllib.request, os, shutil, collections
import torch, numpy as np
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
BASE, REVISION = "BAAI/bge-small-en-v1.5", "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"   # same as config.yaml
CAP, OTHER_CAP_ABCD, MIN_WORDS = 300, 500, 4

In [ ]:
# Mapping (benchmarks/m3/data/bitext_mapping.json) and the Brain's extra Atlas messages (benchmarks/m3/data/atlas_extra_train.json), embedded
BITEXT_MAP = {
 "_note": "Bitext intent -> CallVerse reason, written by the Brain 2026-10-31 for T25b (LoRA v2), to be approved by Yassine before the Colab run. Sources: bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset (ShopNova) and bitext/Bitext-travel-llm-chatbot-training-dataset (Atlas), CDLA-Sharing 1.0, synthetic. Column 'instruction' = message, 'intent' = label. Messages with < 4 words dropped; {{placeholders}} filled. Cap 300 per reason (seed 2026). 'exclude' = ambiguous intents left out (they sit between two reasons). Every other intent -> 'other' (cap 300).",
 "shopnova": {
  "order_status": [
   "track_order",
   "track_delivery",
   "delivery_time"
  ],
  "late_delivery": [
   "delivery_issue"
  ],
  "refund_request": [
   "request_refund",
   "refund_status"
  ],
  "return_item": [
   "return_product",
   "return_product_online"
  ],
  "damaged_item": [
   "damaged_delivery"
  ],
  "exclude": [
   "return_product_in_store",
   "refund_policy",
   "return_policy",
   "missing_item",
   "product_issue",
   "wrong_item",
   "exchange_product",
   "exchange_product_in_store"
  ]
 },
 "atlas": {
  "booking_status": [
   "check_flight_reservation",
   "check_trip_details",
   "check_flight_status"
  ],
  "cancellation": [
   "cancel_flight",
   "cancel_trip"
  ],
  "refund_request": [
   "get_refund"
  ],
  "rebooking": [
   "change_flight",
   "change_trip"
  ],
  "exclude": [
   "check_cancellation_fee",
   "check_arrival_time",
   "check_departure_time"
  ]
 },
 "cap_per_reason": 300,
 "seed": 2026
}
ATLAS_EXTRA = {
 "_note": "AI-GENERATED Atlas TRAINING messages for T25b (LoRA v2), written by Claude (the Brain) 2026-10-31. Only for fine-tuning: not the Intake index examples, not the Atlas val / test sets, not the practice set (none copied or paraphrased from them). Fills reasons that Bitext Travel does not cover.",
 "compensation_request": [
  "My flight landed almost five hours behind schedule, what payment am I entitled to?",
  "I'd like to file a claim for the money owed after my flight was scrapped.",
  "The airline cancelled us two days before departure, is there a cash payment for that?",
  "Can you tell me how to request the delay payout for my trip to Boston?",
  "We sat at the gate for four hours, I believe we are owed something.",
  "Is there a fixed amount you pay when a long-haul flight is very late?",
  "I'm writing to ask for the passenger compensation for my late arrival.",
  "Our connection was missed because of your delay and we arrived six hours late, please compensate us.",
  "What do I need to send you to receive the money for the cancellation?",
  "The flight was called off and I was told I could claim a payment, how?",
  "I want the statutory payment for a delay over three hours.",
  "How long does it take to get the compensation once I submit my claim?",
  "My partner and I both want compensation for the flight that never left.",
  "I was told late arrivals get paid back by distance, how much is mine?",
  "Please start a compensation request for my flight from Denver.",
  "I deserve a payout, the plane arrived at 2 am instead of 9 pm.",
  "Is a cancellation with one week of notice still eligible for compensation?",
  "Can I get money for the hours we lost because of the technical fault?",
  "I'm asking for the delay compensation, not a refund of the ticket.",
  "Your airline partner cancelled my leg, I want what the regulation gives me.",
  "Who handles compensation claims for disrupted flights?",
  "We reached our destination 3 hours 40 minutes late, does that qualify for payment?",
  "I'd like to know the compensation amount for a 2,000 km flight delay.",
  "Please pay us for the cancelled flight, we had to wait until the next day.",
  "Do passengers get a payment when the flight is cancelled for crew reasons?",
  "I lodged a complaint about the delay, when will my compensation be paid?",
  "I am claiming the cancellation payment for the whole family of four.",
  "Is the delay compensation paid in cash or as a voucher?",
  "The long delay made me miss a meeting, I want the compensation I'm owed.",
  "Can you calculate the compensation for my late flight from Miami?",
  "I need the form to claim money for a flight that was badly delayed.",
  "They scrapped our flight after we had boarded, how is the payment worked out?",
  "I think I qualify for a delay payment, could you check?",
  "The flight was more than four hours late, I want to be paid for it.",
  "Tell me what I'm owed for the cancellation, I didn't get any notice.",
  "The outbound leg was a mess, I'd like the money the passenger regulation sets.",
  "My return flight arrived very late, I'd like the payment for that.",
  "Is there compensation if the delay was over three hours on arrival?",
  "Please open a claim for the money owed for my cancelled flight to Atlanta.",
  "I'm owed compensation for the late flight, nobody has contacted me about it."
 ],
 "special_assistance": [
  "My father has Parkinson's, can someone guide him through security?",
  "I'll need a lift to get on board because I can't climb steps.",
  "Is it possible to book an escort for a passenger with dementia?",
  "I'm partially sighted and would like someone to walk me to the gate.",
  "Can my electric wheelchair travel with me and can I get help transferring?",
  "I have a heart condition and can't carry my bags, is there any help?",
  "Please note that I'm hard of hearing and need written boarding information.",
  "At the transfer airport my daughter will need someone to push her chair between gates.",
  "I'm recovering from a hip operation, can I get a buggy to the gate?",
  "Can a member of staff meet my grandfather off the plane?",
  "I'm travelling with oxygen equipment, what help can you arrange?",
  "Do you provide a quiet area or help for a child with autism?",
  "I'm pregnant with twins and need help getting to the plane.",
  "Can you reserve a seat near the toilet for a passenger with mobility problems?",
  "I need assistance with boarding because I have multiple sclerosis.",
  "My aunt is blind, can someone take her from check-in to her seat?",
  "I would like to request mobility assistance for my trip next week.",
  "Is help available for passengers who can't stand in long queues?",
  "I walk with a cane and the gate is far, can I get a wheelchair?",
  "My son is deaf, can the crew be told so they can help him?",
  "Can I bring my service dog in the cabin and get help at boarding?",
  "I have a broken arm and can't manage my luggage, can someone help?",
  "Please arrange wheelchair assistance on arrival for my mother.",
  "I need help with the transfer between terminals, I can't walk far.",
  "Is there special help for elderly passengers flying alone?",
  "I use a prosthetic leg, can I get assistance at security?",
  "We need an ambulift for my husband, he can't use stairs.",
  "Could you inform the hotel that I need an accessible room and help with my bags?",
  "I have a learning disability and would like someone to help me at the airport.",
  "Can staff help my visually impaired friend find the right gate?",
  "I'm asking for special assistance because I have severe arthritis.",
  "My wife had a stroke, she needs help getting on and off the plane.",
  "Can you organise someone to push my wheelchair through the airport?",
  "I need assistance at both ends of my journey due to reduced mobility.",
  "Is there anyone who can help a passenger with a walking frame?",
  "I'd like to add a disability assistance request to my flight booking.",
  "A 90-year-old relative is flying with us, is there an escort service for her?",
  "I am diabetic and may need help during the flight, who should I tell?",
  "Can you help me request a seat with extra room for my leg brace?",
  "I need a wheelchair and someone to help with my bags at arrival."
 ],
 "delay_disruption": [
  "Why is my flight to Seattle showing as delayed?",
  "The departure board says my flight is late, how long will it be?",
  "My flight has been pushed back twice already, what's going on?",
  "Is the morning flight to Chicago going to leave on time? It says delayed.",
  "Nobody has told us anything and departure time passed sixty minutes ago.",
  "The plane is stuck on the tarmac, when will we take off?",
  "My flight was moved to a later time, will I still make my connection?",
  "Is there a storm causing delays on my route today?",
  "I just got a message that my flight is delayed, what are my options?",
  "Our flight is late and I'm going to miss my connecting flight.",
  "Why hasn't boarding started? The flight should have left already.",
  "The airline changed my departure time by three hours.",
  "How late is flight 30412 expected to be?",
  "My flight is delayed because of a technical problem, what happens now?",
  "Is my flight still delayed or has it been cancelled?",
  "There's a delay on my flight, will I get a meal voucher?",
  "My flight to Dallas is running late, can you give me an update?",
  "We were told the crew is late, how long is the delay?",
  "The app shows a new departure time for my flight, is that correct?",
  "My flight is delayed overnight, do I get a hotel?",
  "Flight delays at the airport today, is mine affected?",
  "The airport is closed due to fog, what happens to my flight?",
  "Our flight was diverted to another airport, what do we do now?",
  "My flight keeps getting delayed, I need to know if it will leave tonight.",
  "Is the delay on my flight going to get worse?",
  "I'm at the airport and my flight is delayed by four hours.",
  "My flight was delayed and the gate changed, where do I go?",
  "Can you confirm the new estimated departure of my delayed flight?",
  "Weather has delayed my flight, will you rebook me if I miss my connection?",
  "My flight left late and I'm worried about my connection in Atlanta.",
  "We've been delayed on the plane for two hours, what is the cause?",
  "Is my evening flight affected by the strike?",
  "The flight is late again, is there any information?",
  "My departure is delayed, can I wait in a lounge?",
  "My flight is showing a delay of 90 minutes, should I still come early?",
  "We landed very late and missed the last bus, what support is there?",
  "There's a problem with my flight, it says disruption on the app.",
  "My flight was delayed, will my hotel booking still be fine?",
  "How do I find out the latest status of my delayed flight?",
  "Because of the delay I will arrive after midnight, what help do I get?"
 ]
}

In [ ]:
# 1. ABCD train only (same first-turn rule and subflow mapping as T23)
urllib.request.urlretrieve("https://raw.githubusercontent.com/asappresearch/abcd/master/data/abcd_v1.1.json.gz", "abcd.json.gz")
abcd = json.load(gzip.open("abcd.json.gz", "rt", encoding="utf-8"))
with urllib.request.urlopen("https://raw.githubusercontent.com/yassinemaazaoui114-git/callverse/main/benchmarks/m3/data/abcd_mapping.json") as r:
    abcd_map = json.loads(r.read().decode("utf-8"))
sub2reason = {s: reason for reason, subs in abcd_map.items() if isinstance(subs, list) for s in subs}
def first_turn(chat):
    for who, text in chat["original"]:
        if who == "customer" and len(text.split()) >= MIN_WORDS: return text.strip()
    return None
rows = collections.defaultdict(list)
for chat in abcd["train"]:
    t = first_turn(chat)
    if t: rows["shopnova:" + sub2reason.get(chat["scenario"]["subflow"], "other")].append(t)
random.shuffle(rows["shopnova:other"]); rows["shopnova:other"] = rows["shopnova:other"][:OTHER_CAP_ABCD]
print("ABCD:", {k: len(v) for k, v in rows.items()})

In [ ]:
# 2. Bitext Retail (ShopNova) + Bitext Travel (Atlas), CDLA-Sharing 1.0
from datasets import load_dataset
DIGITS = re.compile(r"number|\bid\b|code|reference", re.I)
def fill(text):   # {{Order Number}} -> 5 digits, other placeholders -> their plain name
    return re.sub(r"\{\{\s*([^}]+?)\s*\}\}", lambda m: str(random.randint(10000, 99999)) if DIGITS.search(m.group(1)) else m.group(1).lower(), text).strip()
def add_bitext(repo, company):
    ds = load_dataset(repo, split="train")
    cmap = BITEXT_MAP[company]
    intent2reason = {i: r for r, ints in cmap.items() if r != "exclude" for i in ints}
    picked = collections.defaultdict(list)
    for ex in ds:
        intent = ex["intent"]
        if intent in cmap["exclude"]: continue
        text = fill(ex["instruction"])
        if len(text.split()) < MIN_WORDS: continue
        picked[intent2reason.get(intent, "other")].append(text)
    for reason, msgs in picked.items():
        random.shuffle(msgs)
        rows[f"{company}:{reason}"] += msgs[:CAP]
    print(company, {r: min(len(m), CAP) for r, m in picked.items()})
add_bitext("bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset", "shopnova")
add_bitext("bitext/Bitext-travel-llm-chatbot-training-dataset", "atlas")
for reason, msgs in ATLAS_EXTRA.items():
    if not reason.startswith("_"): rows["atlas:" + reason] += msgs
labels = sorted(rows); label_id = {l: i for i, l in enumerate(labels)}
data = [(t, label_id[l]) for l in labels for t in rows[l]]
random.shuffle(data)
print(len(data), "rows"); print({l: len(rows[l]) for l in labels})
print("examples:", random.sample(rows["shopnova:order_status"], 2), random.sample(rows["atlas:rebooking"], 2))

In [ ]:
# 3. LoRA adapter (base frozen) + label-based triplet loss; 2 epochs, lr 1e-4, batch 32, warm-up 10 %, fp16
from datasets import Dataset
from peft import LoraConfig, TaskType
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
model = SentenceTransformer(BASE, revision=REVISION)
model.add_adapter(LoraConfig(task_type=TaskType.FEATURE_EXTRACTION, r=16, lora_alpha=32, lora_dropout=0.1,
                             target_modules=["query", "key", "value"]))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad); total = sum(p.numel() for p in model.parameters())
print(f"trainable {trainable:,} of {total:,} ({100 * trainable / total:.2f} %)")
train = Dataset.from_dict({"sentence": [t for t, _ in data], "label": [l for _, l in data]})
args = SentenceTransformerTrainingArguments(
    output_dir="out", num_train_epochs=2, per_device_train_batch_size=32, learning_rate=1e-4,
    warmup_ratio=0.1, fp16=torch.cuda.is_available(), seed=SEED, logging_steps=50, save_strategy="no",
    batch_sampler=BatchSamplers.GROUP_BY_LABEL, report_to="none")
SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=losses.BatchAllTripletLoss(model)).train()

In [ ]:
# 4. Sanity checks on TRAINING data only (not a score)
# (a) adapter ON: same-reason pairs closer than different-reason pairs; (b) adapter OFF must equal the plain base model
sample = random.sample(data, 400); lab = np.array([l for _, l in sample]); texts = [t for t, _ in sample]
def gap(emb):
    sim = emb @ emb.T
    return sim[(lab[:, None] == lab[None, :]) & ~np.eye(len(lab), dtype=bool)].mean(), sim[lab[:, None] != lab[None, :]].mean()
on = model.encode(texts, normalize_embeddings=True)
model.disable_adapters(); off = model.encode(texts, normalize_embeddings=True); model.enable_adapters()
base = SentenceTransformer(BASE, revision=REVISION, device=model.device).encode(texts, normalize_embeddings=True)
print("adapter ON  same / diff: %.3f / %.3f" % gap(on))
print("adapter OFF same / diff: %.3f / %.3f" % gap(off))
print("OFF equals base model:", np.allclose(off, base, atol=1e-4), " max diff", float(np.abs(off - base).max()))

In [ ]:
# 5. Save the ADAPTER only + training info, zip for download
OUT = "bge-small-callverse-lora"
model.save(OUT)
json.dump({"base": BASE, "revision": REVISION, "method": "LoRA r16 alpha32 dropout0.1 query/key/value", "train_rows": len(data),
           "per_label": {l: len(rows[l]) for l in labels}, "seed": SEED, "epochs": 2, "batch": 32, "lr": 1e-4,
           "loss": "BatchAllTripletLoss (group-by-label batches)",
           "data": "ABCD train (MIT, other capped 500) + Bitext Retail + Bitext Travel (CDLA-Sharing 1.0, 300/reason) + 120 Brain Atlas messages; CallVerse index examples NOT used",
           "use": "adapter ON for Intake, OFF for rule search"}, open(f"{OUT}/callverse_training.json", "w"), indent=1)
print(sorted(os.listdir(OUT)))
reload = SentenceTransformer(OUT)   # check it loads back as base + adapter
print("reload OK, same output:", np.allclose(reload.encode(texts[:20], normalize_embeddings=True), on[:20], atol=1e-4))
shutil.make_archive(OUT, "zip", OUT)
print(os.path.getsize(OUT + ".zip") / 1e6, "MB")
from google.colab import files; files.download(OUT + ".zip")